# Getting Started with PointillSim

This quick-start guide shows you how to:
1. Load pre-generated sample datasets
2. Generate your own simulated data in just a few lines
3. Visualize the results

For a deeper understanding of the framework, see `01_simulation_framework.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

---
## 1. Loading Pre-generated Sample Datasets

The fastest way to get started is to load one of the bundled sample datasets.

In [ ]:
from pointillsim import load_sample, list_samples

# See what's available
print("Available sample datasets:")
for name in list_samples():
    print(f"  - {name}")

In [ ]:
# Load a sample dataset
fov, tissue, dots_df = load_sample("simple_fov")

print(f"FOV: {len(fov.cell_centroids)} cells")
print(f"Tissue: {tissue.n_genes} genes, {tissue.n_cell_types} cell types")
print(f"Dots: {len(dots_df)} transcripts")

In [ ]:
# Visualize the cells colored by type
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Cell positions
ax = axes[0]
scatter = ax.scatter(
    fov.cell_centroids[:, 0],
    fov.cell_centroids[:, 1],
    c=fov.class_instance,
    cmap='Set1',
    s=20, alpha=0.7
)
ax.set_aspect('equal')
ax.set_title('Cell Positions by Type')
plt.colorbar(scatter, ax=ax, label='Cell Type')

# Transcript dots
ax = axes[1]
ax.scatter(dots_df['x'], dots_df['y'], s=1, alpha=0.3, c='red')
ax.set_aspect('equal')
ax.set_title(f'Transcript Dots ({len(dots_df):,} total)')

plt.tight_layout()
plt.show()

In [ ]:
# Look at the dots DataFrame structure
print("Dots DataFrame:")
print(dots_df.head(10))
print(f"\nColumns: {list(dots_df.columns)}")

In [ ]:
# Try a more complex sample - cortex-like layered tissue
fov_cortex, tissue_cortex, dots_cortex = load_sample("cortex_like")

fig, ax = plt.subplots(figsize=(10, 10))
scatter = ax.scatter(
    fov_cortex.cell_centroids[:, 0],
    fov_cortex.cell_centroids[:, 1],
    c=fov_cortex.class_instance,
    cmap='Set1',
    s=15, alpha=0.6
)
ax.set_aspect('equal')
ax.set_title(f'Cortex-like Tissue\n({len(fov_cortex.cell_centroids)} cells, {tissue_cortex.n_cell_types} types)')
plt.colorbar(scatter, ax=ax, label='Cell Type')
plt.show()

# Notice the layered structure - cell types vary with Y position

---
## 2. Generating Your Own Data (Minimal Example)

Here's the simplest way to generate simulated spatial transcriptomics data.

In [ ]:
from pointillsim import (
    TissueCellTypes,
    CellTypesProperties,
    HybISS_Setup,
    FOVDistribution,
    FrameWideElement,
    RandomCellTypeRule,
)

In [ ]:
# Step 1: Define tissue expression profiles
tissue = TissueCellTypes()
tissue.generate_types_and_markers(
    n_genes=50,       # Number of genes in panel
    n_cell_types=5,   # Number of cell types
)

# Step 2: Define cell morphology
cell_props = CellTypesProperties(n_cell_types=5)

# Step 3: Create FOV generator
fov_dist = FOVDistribution(
    frame_size=800,
    background_element=lambda: FrameWideElement(
        frame_size=800,
        tipical_cell_spacing=18,
        rules=RandomCellTypeRule(n_cell_types=5)
    ),
)

# Step 4: Generate FOV and apply morphology
fov = fov_dist.generate_fov()
cell_props.apply(fov)

# Step 5: Generate transcript observations
hybiss = HybISS_Setup(tissue)
hybiss.observe_dots(fov)

# Step 6: Export
dots_df = hybiss.make_pandas_df()
cells_df = fov.make_pandas_df()

print(f"Generated {len(cells_df)} cells with {len(dots_df)} transcript dots")

In [ ]:
# Visualize
from pointillsim import plot_fov

fig, axes = plot_fov(
    fov, 
    tissue,
    dots_df=dots_df,
    figsize=(16, 4)
)
plt.show()

---
## 3. Understanding the Output

PointillSim produces two main outputs:

### Cells DataFrame (`cells_df`)
Contains ground truth information about each cell.

In [ ]:
print("Cells DataFrame columns:")
print(cells_df.columns.tolist())
print(f"\nShape: {cells_df.shape}")
print("\nFirst 5 rows:")
cells_df.head()

Key columns:
- `X`, `Y`: Cell centroid coordinates
- `Class ID`: Realized cell type (integer)
- `Class 0`, `Class 1`, ...: One-hot encoded cell type
- `ProbClass0`, `ProbClass1`, ...: Soft probabilities (ground truth)
- `Minor Axis`, `Major Axis`, `Rotation`: Cell morphology
- `RNA Concentration`: Relative RNA content

### Dots DataFrame (`dots_df`)
Contains observed transcript positions.

In [ ]:
print("Dots DataFrame columns:")
print(dots_df.columns.tolist())
print(f"\nShape: {dots_df.shape}")
print("\nFirst 10 rows:")
dots_df.head(10)

Key columns:
- `x`, `y`: Transcript dot coordinates
- `gene`: Gene name
- `cell`: Index of the cell this transcript belongs to

In [ ]:
# Gene counts per cell type
import pandas as pd

dots_with_type = dots_df.merge(
    cells_df[['Class ID']].reset_index().rename(columns={'index': 'cell'}),
    on='cell'
)

# Pivot to get counts matrix
gene_counts = dots_with_type.groupby(['Class ID', 'gene']).size().unstack(fill_value=0)
print("Transcripts per cell type (first 10 genes):")
gene_counts.iloc[:, :10]

---
## 4. Export to AnnData/SpatialData

For integration with standard single-cell analysis tools.

In [ ]:
# Export to AnnData (if anndata is installed)
try:
    adata = fov.to_anndata(tissue)
    print(f"AnnData object: {adata}")
    print(f"  obs (cells): {adata.obs.shape}")
    print(f"  var (genes): {adata.var.shape}")
    print(f"  spatial coordinates in: obsm['spatial']")
except ImportError:
    print("Install anndata to enable AnnData export: pip install anndata")

---
## Next Steps

Now that you've seen the basics, explore these notebooks for more details:

1. **01_simulation_framework.ipynb**: Deep dive into the simulation philosophy and all components
2. **02_elements_and_rules.ipynb**: Creating complex tissue structures with different cell type patterns
3. **03_batch_generation.ipynb**: Generating multiple FOVs and datasets
4. **04_real_expression_data.ipynb**: Using real scRNA-seq data as input